# AnyUp 3D Training Notebook

This notebook trains the 3D `AnyUp` model (video feature upsampling).  
It mirrors the structure of `train.py` but is interactive — tweak and re-run cells freely.

**Branch:** `3Dconv`  
**Model:** `anyup.model.AnyUp` (spatiotemporal cross-attention)  
**Teacher:** VideoMAE (or any video encoder producing 5D feature tensors `(B, C, T, H, W)`)

## 0. Imports & Paths

In [ ]:
import os
import sys
import random
import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import autocast
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

# Make sure the repo root is on the path
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Configuration

All knobs are here. Adjust before running the rest.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

@dataclass
class TStage:
    t: int           # number of frames for this stage
    start_step: int  # global step at which this stage activates
    batch_size: int  # batch size to use in this stage

@dataclass
class TrainConfig:
    # --- Paths ---
    model_ckpt_2d: Optional[str] = "checkpoints/anyup_2d.pth"  # pretrained 2D weights; None = random init
    checkpoint_dir: str = "checkpoints/anyup3d"
    resume: Optional[str] = None    # path to a 3D checkpoint to resume; None = fresh start
    log_dir: str = "runs/anyup3d"

    # --- Video encoder (teacher) ---
    encoder: str = "videomae"                        # only 'videomae' implemented below
    encoder_model: str = "MCG-NJU/videomae-base"     # HuggingFace model id

    # --- Model hyper-params ---
    input_dim: int = 3
    qk_dim: int = 128
    num_heads: int = 4
    window_ratio: float = 0.1
    kernel_size_lfu: int = 5
    t_k: int = 1              # temporal kernel size for 3D convs

    # --- Spatial resolution ---
    img_size: int = 224        # spatial size fed to the model
    crop_size: int = 448       # high-res crop for GT token extraction

    # --- T-curriculum (frame count ramps up over training) ---
    t_schedule: List[TStage] = field(default_factory=lambda: [
        TStage(t=1,  start_step=0,     batch_size=8),
        TStage(t=4,  start_step=5000,  batch_size=4),
        TStage(t=8,  start_step=15000, batch_size=2),
        TStage(t=16, start_step=30000, batch_size=1),
    ])

    # --- Optimizer ---
    lr: float = 2e-4
    weight_decay: float = 1e-4
    grad_clip_max_norm: float = 1.0

    # --- Loss weights ---
    lambda_cos_mse: float = 1.0
    lambda_input_consistency: float = 0.1
    lambda_self_consistency: float = 0.1
    lambda_temporal_consistency: float = 0.2
    temporal_lambda_warmup_steps: int = 5000

    # --- Training duration ---
    max_steps: int = 50_000
    save_every_n_steps: int = 500
    val_every_n_steps: int = 1000
    log_every_n_steps: int = 50

    # --- Data ---
    dataset_root: str = "data/videos"
    num_workers: int = 4
    pin_memory: bool = True

    # --- Misc ---
    seed: int = 42
    debug: bool = False
    debug_steps: int = 10


cfg = TrainConfig()

# Quick-override for a fast smoke test:
# cfg.debug = True

print(cfg)

## 2. Reproducibility & Device

In [ ]:
import numpy as np

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

## 3. Logger (TensorBoard)

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_log_dir = Path(cfg.log_dir) / timestamp
run_log_dir.mkdir(parents=True, exist_ok=True)

writer = SummaryWriter(log_dir=str(run_log_dir))
print(f"TensorBoard log dir: {run_log_dir}")
print("Start TensorBoard with:  tensorboard --logdir", cfg.log_dir)

## 4. Video Encoder (Teacher / GT features)

The teacher extracts ground-truth spatiotemporal features that AnyUp is trained to reproduce.  
Currently wired for **VideoMAE**. Swap this cell to use a different encoder.

In [ ]:
# ---- VideoMAE encoder ----
# Output shape: (B, T*H*W, C) spatial tokens -> reshaped to (B, C, T, H, W)
# For ViT-B/16 at 224px: H=W=14, so token grid is (T, 14, 14)

from transformers import VideoMAEModel, AutoFeatureExtractor

print(f"Loading teacher: {cfg.encoder_model} ...")
feature_extractor = AutoFeatureExtractor.from_pretrained(cfg.encoder_model)
encoder = VideoMAEModel.from_pretrained(cfg.encoder_model).to(device).eval()

# Freeze — we never train the teacher
for p in encoder.parameters():
    p.requires_grad_(False)

# Derive encoder output dim & patch size
ENCODER_DIM = encoder.config.hidden_size     # e.g. 768 for ViT-B
PATCH_SIZE  = encoder.config.patch_size      # e.g. 16
TOKEN_H = cfg.img_size // PATCH_SIZE
TOKEN_W = cfg.img_size // PATCH_SIZE

print(f"Encoder dim={ENCODER_DIM}, patch_size={PATCH_SIZE}, spatial token grid={TOKEN_H}x{TOKEN_W}")

## 5. AnyUp 3D Model

In [ ]:
from anyup.model import AnyUp

anyup = AnyUp(
    input_dim=cfg.input_dim,
    qk_dim=cfg.qk_dim,
    num_heads=cfg.num_heads,
    window_ratio=cfg.window_ratio,
    kernel_size_lfu=cfg.kernel_size_lfu,
    t_k=cfg.t_k,
).to(device)

# --- Optional: load 2D pretrained weights ---
if cfg.model_ckpt_2d and Path(cfg.model_ckpt_2d).exists():
    from anyup.utils.training import load_2d_weights  # implement below if needed
    print(f"Loading 2D checkpoint: {cfg.model_ckpt_2d}")
    state = torch.load(cfg.model_ckpt_2d, map_location="cpu", weights_only=False)
    # state dict key may be nested
    sd = state.get("anyup", state)
    sd = sd.get("state_dict", sd)
    missing, unexpected = anyup.load_state_dict(sd, strict=False)
    print(f"  Missing keys : {len(missing)}")
    print(f"  Unexpected   : {len(unexpected)}")
elif cfg.model_ckpt_2d:
    print(f"[WARNING] 2D checkpoint not found at '{cfg.model_ckpt_2d}' — starting from random init")
else:
    print("Starting from random init (no 2D checkpoint configured)")

# --- Optional: resume 3D checkpoint ---
global_step = 0
if cfg.resume and Path(cfg.resume).exists():
    print(f"Resuming from: {cfg.resume}")
    ckpt = torch.load(cfg.resume, map_location="cpu", weights_only=False)
    anyup.load_state_dict(ckpt["anyup"])
    global_step = ckpt.get("global_step", 0)
    print(f"  Resumed at step {global_step}")

anyup.train()
num_params = sum(p.numel() for p in anyup.parameters() if p.requires_grad)
print(f"AnyUp 3D trainable params: {num_params:,}")

## 6. Loss Functions

In [ ]:
from anyup.loss import Cosine_MSE

criterion = Cosine_MSE()

def cosine_mse_3d(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """
    pred, target: (B, C, T, H, W)  ->  flatten T,H,W then call 2D criterion.
    Returns scalar loss.
    """
    B, C, T, H, W = pred.shape
    # Treat T*H*W as a spatial 'area' by merging T into H
    pred_2d   = pred.view(B, C, T * H, W)
    target_2d = target.view(B, C, T * H, W)
    return criterion(pred_2d, target_2d)["total"]


def get_temporal_lambda(step: int) -> float:
    """Linearly ramp temporal consistency weight from 0 to cfg value."""
    warmup = cfg.temporal_lambda_warmup_steps
    if warmup <= 0:
        return cfg.lambda_temporal_consistency
    return cfg.lambda_temporal_consistency * min(1.0, step / warmup)


print("Loss functions ready.")

## 7. Dataset & DataLoader

**TODO:** Replace the stub below with your actual `VideoDataset`.  
The dataset `__getitem__` should return a dict with at least:
```python
{
    "video": torch.Tensor,   # (T, C, H, W) float32, ImageNet-normalized
}
```
Optionally `"augmented_video"` for consistency regularization.

In [ ]:
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.v2 as T
from torchvision.transforms import InterpolationMode


# ============================================================
# Stub dataset — replace this with your real VideoDataset
# ============================================================
class StubVideoDataset(Dataset):
    """
    Generates random (T, C, H, W) tensors for smoke-testing.
    Replace with a real dataset that reads from cfg.dataset_root.
    """
    def __init__(self, num_samples: int = 1000, t: int = 1,
                 img_size: int = 224, augment: bool = True):
        self.num_samples = num_samples
        self.t = t
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # ImageNet-normalized random frame(s)
        video = torch.randn(self.t, 3, self.img_size, self.img_size)
        sample = {"video": video}
        if self.augment:
            sample["augmented_video"] = video + 0.05 * torch.randn_like(video)
        return sample


# We'll build the actual loader inside the training loop
# because batch_size and T change with the curriculum.
def build_loader(t: int, batch_size: int, split: str = "train") -> DataLoader:
    """
    Build a DataLoader for a given T and batch_size.
    Swap StubVideoDataset for your real dataset here.
    """
    dataset = StubVideoDataset(
        num_samples=500 if cfg.debug else 50_000,
        t=t,
        img_size=cfg.img_size,
        augment=True,
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(split == "train"),
        num_workers=cfg.num_workers if not cfg.debug else 0,
        pin_memory=cfg.pin_memory and device.type == "cuda",
        drop_last=True,
    )
    return loader


# Quick sanity check
test_loader = build_loader(t=1, batch_size=2)
sample_batch = next(iter(test_loader))
print("Sample batch shapes:")
for k, v in sample_batch.items():
    print(f"  {k}: {v.shape}")

## 8. Optimizer

In [ ]:
optimizer = torch.optim.AdamW(
    anyup.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

# Cosine LR decay over the full training run
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.max_steps,
    eta_min=cfg.lr * 0.01,
)

if cfg.resume and Path(cfg.resume).exists():
    ckpt = torch.load(cfg.resume, map_location="cpu", weights_only=False)
    if "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])

print("Optimizer and scheduler ready.")

## 9. Checkpoint Helpers

In [ ]:
ckpt_dir = Path(cfg.checkpoint_dir)
ckpt_dir.mkdir(parents=True, exist_ok=True)

def save_checkpoint(step: int, tag: str = ""):
    name = f"anyup3d_step{step}{('_' + tag) if tag else ''}.pth"
    path = ckpt_dir / name
    torch.save({
        "anyup": anyup.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "global_step": step,
        "cfg": vars(cfg),
    }, path)
    print(f"Saved checkpoint: {path}")
    return path

print(f"Checkpoints will be saved to: {ckpt_dir}")

## 10. Training Loop

The loop implements:
1. **T-curriculum** — batch size and frame count change at milestone steps
2. **Primary loss** — `Cosine_MSE` between AnyUp output and teacher GT features
3. **Input-consistency** — same video, different resolution → features should match
4. **Self-consistency** — augmented video → features should be close to clean
5. **Temporal consistency** — adjacent frames → features should be smooth (ramped in)

In [ ]:
# ---- Utility: extract teacher features from a video batch ----
@torch.no_grad()
def extract_teacher_features(video: torch.Tensor) -> torch.Tensor:
    """
    video : (B, T, C, H, W) float32, ImageNet-normalized
    returns: (B, C_enc, T, H_tok, W_tok)  — 5D feature volume
    """
    B, T, C, H, W = video.shape

    # VideoMAE expects (B, C, T, H, W) — reorder
    pixel_values = video.permute(0, 2, 1, 3, 4).to(device)  # (B, C, T, H, W)

    out = encoder(pixel_values=pixel_values)
    # last_hidden_state: (B, N_tokens, C_enc)  where N = T * H_tok * W_tok
    tokens = out.last_hidden_state  # (B, N, C_enc)

    N = tokens.shape[1]
    H_tok = W_tok = int((N / T) ** 0.5)
    assert H_tok * W_tok * T == N, f"Cannot reshape {N} tokens into ({T}, {H_tok}, {W_tok})"

    # (B, T, H_tok, W_tok, C) -> (B, C, T, H_tok, W_tok)
    feats = tokens.reshape(B, T, H_tok, W_tok, -1).permute(0, 4, 1, 2, 3).contiguous()
    return feats


# ---- Utility: determine current curriculum stage ----
def get_current_stage(step: int) -> TStage:
    stage = cfg.t_schedule[0]
    for s in cfg.t_schedule:
        if step >= s.start_step:
            stage = s
    return stage


print("Training utilities ready.")

In [ ]:
# ---- Main training loop ----

max_steps = cfg.debug_steps if cfg.debug else cfg.max_steps
current_stage = get_current_stage(global_step)
loader = build_loader(t=current_stage.t, batch_size=current_stage.batch_size)
loader_iter = iter(loader)

anyup.train()

pbar = tqdm(range(global_step, max_steps), initial=global_step, total=max_steps, desc="Training")

for step in pbar:

    # ---- Check if curriculum stage changed ----
    new_stage = get_current_stage(step)
    if new_stage.t != current_stage.t or new_stage.batch_size != current_stage.batch_size:
        print(f"\n[Step {step}] Curriculum: T={current_stage.t}->{new_stage.t}, "
              f"BS={current_stage.batch_size}->{new_stage.batch_size}")
        current_stage = new_stage
        loader = build_loader(t=current_stage.t, batch_size=current_stage.batch_size)
        loader_iter = iter(loader)

    # ---- Get batch (reset iterator if exhausted) ----
    try:
        batch = next(loader_iter)
    except StopIteration:
        loader_iter = iter(loader)
        batch = next(loader_iter)

    video = batch["video"].to(device)           # (B, T, C, H, W)
    aug_video = batch.get("augmented_video")     # may be None
    if aug_video is not None:
        aug_video = aug_video.to(device)

    B, T, C, H, W = video.shape

    # ---- Forward pass ----
    with autocast(device_type=device.type, dtype=torch.bfloat16):

        # GT features from frozen teacher
        gt_feats = extract_teacher_features(video)   # (B, C_enc, T, H_tok, W_tok)

        # AnyUp input: video as (B, C, T, H, W)
        # We pass the LR features (gt_feats at native resolution) and ask AnyUp to
        # upsample to the token grid size of the HR crop.
        video_5d = video.permute(0, 2, 1, 3, 4)     # (B, C, T, H, W)
        out_size = (gt_feats.shape[-2], gt_feats.shape[-1])  # (H_tok, W_tok)

        pred_feats = anyup(video_5d, gt_feats, output_size=out_size)  # (B, C_enc, T, H_tok, W_tok)

        # 1. Primary reconstruction loss
        loss_rec = cosine_mse_3d(pred_feats, gt_feats) * cfg.lambda_cos_mse

        # 2. Self-consistency under augmentation
        loss_aug = torch.tensor(0.0, device=device)
        if aug_video is not None and cfg.lambda_self_consistency > 0:
            aug_5d = aug_video.permute(0, 2, 1, 3, 4)
            with torch.no_grad():
                gt_feats_aug = extract_teacher_features(aug_video)
            pred_aug = anyup(aug_5d, gt_feats_aug, output_size=out_size)
            loss_aug = cosine_mse_3d(pred_aug, pred_feats.detach()) * cfg.lambda_self_consistency

        # 3. Temporal consistency (adjacent frames should have smooth features)
        loss_temp = torch.tensor(0.0, device=device)
        t_lam = get_temporal_lambda(step)
        if T > 1 and t_lam > 0:
            diff = pred_feats[:, :, 1:] - pred_feats[:, :, :-1]  # (B, C, T-1, H, W)
            loss_temp = diff.pow(2).mean() * t_lam

        total_loss = loss_rec + loss_aug + loss_temp

    # ---- Backward ----
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(anyup.parameters(), cfg.grad_clip_max_norm)
    optimizer.step()
    scheduler.step()

    # ---- Logging ----
    if step % cfg.log_every_n_steps == 0:
        lr = optimizer.param_groups[0]["lr"]
        writer.add_scalar("loss/total",       total_loss.item(), step)
        writer.add_scalar("loss/rec",         loss_rec.item(),   step)
        writer.add_scalar("loss/aug",         loss_aug.item(),   step)
        writer.add_scalar("loss/temporal",    loss_temp.item(),  step)
        writer.add_scalar("lr",               lr,                step)
        writer.add_scalar("curriculum/T",     current_stage.t,   step)
        writer.add_scalar("curriculum/bs",    current_stage.batch_size, step)

        pbar.set_postfix({
            "loss": f"{total_loss.item():.4f}",
            "rec":  f"{loss_rec.item():.4f}",
            "T":    current_stage.t,
            "lr":   f"{lr:.2e}",
        })

    # ---- Checkpoint ----
    if (step + 1) % cfg.save_every_n_steps == 0:
        save_checkpoint(step + 1)

    if cfg.debug and step >= cfg.debug_steps - 1:
        print(f"Debug mode: stopping after {cfg.debug_steps} steps.")
        break


# Final checkpoint
save_checkpoint(max_steps, tag="final")
writer.flush()
writer.close()
print("Training complete.")

## 11. Quick Validation / Smoke Test

Run this at any point to check the model produces sensible outputs without running the full training loop.

In [ ]:
anyup.eval()

with torch.no_grad():
    dummy_video = torch.randn(1, 4, 3, cfg.img_size, cfg.img_size, device=device)  # B=1, T=4
    dummy_feats = torch.randn(1, ENCODER_DIM, 4, TOKEN_H, TOKEN_W, device=device)
    dummy_video_5d = dummy_video.permute(0, 2, 1, 3, 4)

    out = anyup(dummy_video_5d, dummy_feats, output_size=(TOKEN_H, TOKEN_W))
    print(f"Input  features : {dummy_feats.shape}")
    print(f"Output features : {out.shape}")
    assert out.shape == dummy_feats.shape, "Shape mismatch!"
    print("Validation passed.")

anyup.train()